In [ ]:
# Environment setup & Imports
import os, sys
sys.path.append(os.path.abspath('..'))
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from IPython.display import display, HTML

from shared_utils.theme import PALETTE, apply_sleek_theme, create_dashboard_header
from chua_circuit import simulate_chua_master, simulate_pecora_carroll_masking
from dripping_faucet import simulate_dripping_faucet
from aref_blinking_vortex import simulate_blinking_vortex_mixing

apply_sleek_theme()
display(HTML(create_dashboard_header(
    'ACT 06: PHYSICAL HARDWARE, CIRCUIT CHAOS & HYDRODYNAMICS',
    'Interactive GUI Suite: Chua Double Scroll, Secure Comms, Dripping Faucet & Chaotic Mixing'
)))

In [ ]:
# LAB 6.1: CHUA's DOUBLE SCROLL CIRCUIT & SECURE MASKING
display(HTML(create_dashboard_header(
    'LAB 6.1: CHUA CIRCUIT DOUBLE SCROLL & PECORA-CARROLL COMMS',
    'Scrub parameter alpha to observe Hopf breakdown into double scroll, and test chaotic signal recovery.'
)))

def render_chua_dashboard(alpha_param, beta_param):
    t_pts, xm, ym, zm = simulate_chua_master(alpha=alpha_param, beta=beta_param, t_span=(0.0, 45.0))
    t_ev, x_raw, x_trans, msg_orig, msg_rec = simulate_pecora_carroll_masking(alpha=alpha_param, beta=beta_param)
    
    fig, axs = plt.subplots(1, 2, figsize=(13, 5), dpi=100)
    
    # Panel A: Chua Double Scroll Attractor (x vs z)
    axs[0].plot(xm, zm, color=PALETTE['cyan'], lw=0.6, alpha=0.85)
    axs[0].scatter([0], [0], color=PALETTE['crimson'], s=50, label='Saddle-Focus D0')
    axs[0].set_title('Chua Double Scroll Attractor (x vs z) | alpha = ' + str(round(alpha_param, 2)), fontsize=11)
    axs[0].set_xlabel('Capacitor C1 Voltage x', fontsize=10)
    axs[0].set_ylabel('Inductor Current z', fontsize=10)
    axs[0].legend(loc='upper right')
    
    # Panel B: Pecora-Carroll Chaotic Masking & Message Recovery
    axs[1].plot(t_ev[:400], msg_orig[:400], color=PALETTE['emerald'], lw=2.0, label='Hidden Message msg(t)')
    axs[1].plot(t_ev[:400], x_trans[:400], color=PALETTE['border'], lw=0.8, alpha=0.6, label='Transmitted Carrier s(t)')
    axs[1].plot(t_ev[:400], msg_rec[:400], '--', color=PALETTE['amber'], lw=1.5, label='Recovered Message (Slave)')
    axs[1].set_title('Pecora-Carroll Secure Synchronization Recovery', fontsize=11)
    axs[1].set_xlabel('Time t', fontsize=10)
    axs[1].set_ylabel('Signal Amplitude', fontsize=10)
    axs[1].legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()

s_alpha = widgets.FloatSlider(value=15.6, min=8.0, max=18.0, step=0.2, description='Alpha (C2/C1):', layout=widgets.Layout(width='380px'))
s_beta = widgets.FloatSlider(value=28.0, min=15.0, max=35.0, step=0.5, description='Beta (L):', layout=widgets.Layout(width='380px'))
ui_chua = widgets.HBox([s_alpha, s_beta], layout=widgets.Layout(padding='10px', background_color=PALETTE['surface'], border='1px solid ' + PALETTE['border'], border_radius='8px'))
out_chua = widgets.interactive_output(render_chua_dashboard, {'alpha_param': s_alpha, 'beta_param': s_beta})
display(ui_chua, out_chua)

In [ ]:
# LAB 6.2: SHAW DRIPPING FAUCET & POINCARE RETURN MAP
display(HTML(create_dashboard_header(
    'LAB 6.2: SHAW DRIPPING FAUCET CAPILLARY PINCH-OFF',
    'Scrub flow rate Q to track droplet cascade: Period-1, Period-2, and chaotic multi-band strange attractors.'
)))

def render_faucet_dashboard(flow_q):
    dt_series, dt_n, dt_next = simulate_dripping_faucet(flow_rate=flow_q, n_drops=400)
    
    fig, axs = plt.subplots(1, 2, figsize=(13, 5), dpi=100)
    
    # Panel A: Inter-drop intervals sequence
    axs[0].plot(range(len(dt_series)), dt_series, 'o-', color=PALETTE['cyan'], markersize=3, lw=0.8)
    axs[0].set_title('Inter-Drop Time Interval Sequence delta_t_n | Flow Q = ' + str(round(flow_q, 2)), fontsize=11)
    axs[0].set_xlabel('Drop Number n', fontsize=10)
    axs[0].set_ylabel('Time to Next Drop (s)', fontsize=10)
    
    # Panel B: 2D Poincaré Return Map (delta_t_n vs delta_t_{n+1})
    axs[1].scatter(dt_n, dt_next, s=16, color=PALETTE['amber'], edgecolors=PALETTE['crimson'], lw=0.5)
    axs[1].set_title('Poincare Map (delta_t_n vs delta_t_{n+1}) | Attractor Topology', fontsize=11)
    axs[1].set_xlabel('Interval delta_t_n (s)', fontsize=10)
    axs[1].set_ylabel('Interval delta_t_{n+1} (s)', fontsize=10)
    axs[1].grid(True, linestyle='--', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

s_flow = widgets.FloatSlider(value=0.85, min=0.2, max=2.0, step=0.05, description='Flow Rate Q:', layout=widgets.Layout(width='450px'))
ui_faucet = widgets.HBox([s_flow], layout=widgets.Layout(padding='10px', background_color=PALETTE['surface'], border='1px solid ' + PALETTE['border'], border_radius='8px'))
out_faucet = widgets.interactive_output(render_faucet_dashboard, {'flow_q': s_flow})
display(ui_faucet, out_faucet)

In [ ]:
# LAB 6.3: AREF'S BLINKING VORTEX CHAOTIC ADVECTION
display(HTML(create_dashboard_header(
    'LAB 6.3: AREF BLINKING VORTEX CHAOTIC ADVECTION',
    'Step through blinking cycles: Witness Smale horseshoe stretching and folding in laminar Stokes flow.'
)))

hist_x, hist_y = simulate_blinking_vortex_mixing(n_particles=300, n_cycles=12)

def render_mixing_dashboard(cycle_step):
    fig, ax = plt.subplots(figsize=(7.5, 7.5), dpi=100)
    # Draw circular container
    circ = plt.Circle((0, 0), 1.2, color=PALETTE['border'], fill=False, linestyle='--', lw=1.2)
    ax.add_patch(circ)
    
    # Vortex centers
    ax.scatter([-0.6], [0], color=PALETTE['crimson'], s=80, marker='x', label='Left Vortex')
    ax.scatter([0.6], [0], color=PALETTE['amber'], s=80, marker='x', label='Right Vortex')
    
    # Advected dye particles
    ax.scatter(hist_x[cycle_step], hist_y[cycle_step], s=18, color=PALETTE['cyan'], edgecolors=PALETTE['text_main'], lw=0.4, label='Dye Tracer Particles')
    
    ax.set_title('Aref Chaotic Advection | Cycle ' + str(cycle_step) + ' / 12 (Horseshoe Folding)', fontsize=11, pad=12)
    ax.set_xlabel('Fluid Domain x', fontsize=10)
    ax.set_ylabel('Fluid Domain y', fontsize=10)
    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-1.3, 1.3)
    ax.set_aspect('equal')
    ax.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

s_cycle = widgets.IntSlider(value=0, min=0, max=12, step=1, description='Blink Cycle:', layout=widgets.Layout(width='400px'))
ui_mix = widgets.HBox([s_cycle], layout=widgets.Layout(padding='10px', background_color=PALETTE['surface'], border='1px solid ' + PALETTE['border'], border_radius='8px'))
out_mix = widgets.interactive_output(render_mixing_dashboard, {'cycle_step': s_cycle})
display(ui_mix, out_mix)